In [2]:
import pandas as pd


In [6]:
import torch
from sentence_transformers import SentenceTransformer

print(f"Using GPU: {torch.cuda.is_available()}")
model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
print("Model loaded on RTX 4050 successfully!")

Using GPU: True


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4437.87it/s]


Model loaded on RTX 4050 successfully!


In [3]:
df = pd.read_csv('dataset/synthetic_logs.csv')
df

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert
...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert
2407,2025-08-03 03:07:47,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert


In [4]:
df.source.unique()

<StringArray>
[      'ModernCRM', 'AnalyticsEngine',        'ModernHR',   'BillingSystem',
   'ThirdPartyAPI',       'LegacyCRM']
Length: 6, dtype: str

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN 
import numpy as np

c:\Users\shifa\Desktop\Projects\classification_logs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:

embeddings = model.encode(df['log_message'].tolist())
embeddings[:2]

array([[-1.02939658e-01,  3.35459076e-02, -2.20260322e-02,
         1.55102310e-03, -9.86923371e-03, -1.78956211e-01,
        -6.34410307e-02, -6.01761937e-02,  2.81108450e-02,
         5.99620566e-02, -1.72618628e-02,  1.43368531e-03,
        -1.49560064e-01,  3.15292412e-03, -5.66030703e-02,
         2.71686260e-02, -1.49890119e-02, -3.54037806e-02,
        -3.62936333e-02, -1.45409480e-02, -5.61491353e-03,
         8.75538364e-02,  4.55120653e-02,  2.50963047e-02,
         1.00187324e-02,  1.24266408e-02, -1.39923513e-01,
         7.68695921e-02,  3.14095058e-02, -4.15245118e-03,
         4.36902493e-02,  1.71249919e-02, -8.00950900e-02,
         5.74005917e-02,  1.89092699e-02,  8.55261907e-02,
         3.96399312e-02, -1.34371817e-01, -1.44365174e-03,
         3.06709413e-03,  1.76854104e-01,  4.44887439e-03,
        -1.69274583e-02,  2.24267114e-02, -4.35049571e-02,
         6.09026384e-03, -9.98170208e-03, -6.23972826e-02,
         1.07372338e-02, -6.04890706e-03, -7.14661106e-0

In [1]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not detected by PyTorch.")

PyTorch Version: 2.5.1+cu121
CUDA Available: True
GPU Name: NVIDIA GeForce RTX 4050 Laptop GPU


In [11]:
dbscan = DBSCAN(eps=0.2,min_samples =2,metric='cosine')
clusters = dbscan.fit_predict(embeddings)

df['cluster'] = clusters

In [13]:
df.cluster.unique()
df[df.cluster == 1]

,timestamp,source,log_message,target_label,complexity,cluster
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
10,8/9/2025 18:58,ModernCRM,Email server encountered a sending fault,Error,bert,1
217,1/22/2025 5:45,BillingSystem,Mail service encountered a delivery glitch,Error,bert,1
248,5/2/2025 23:04,ModernHR,Service disruption caused by email sending error,Critical Error,bert,1
265,3/30/2025 23:53,ModernCRM,Email system had a problem sending emails,Error,bert,1
361,11/19/2025 23:06,BillingSystem,Email service experienced a sending issue,Error,bert,1
450,10/27/2025 5:59,ThirdPartyAPI,Email delivery system encountered an error,Error,bert,1
477,12/2/2025 10:30,AnalyticsEngine,Email transmission error caused service impact,Critical Error,bert,1
570,11/7/2025 18:08,ThirdPartyAPI,Email service impacted by sending failure,Critical Error,bert,1
678,4/28/2025 15:13,AnalyticsEngine,Email delivery problem affected system,Critical Error,bert,1


In [14]:
cluster_counts = df['cluster'].value_counts()
large_clusters = cluster_counts[cluster_counts>10].index

for cluster in large_clusters:
    print(f"Cluster {cluster}:")
    print(df[df['cluster'] == cluster]['log_message'].head(5).to_string(index = False))
    print()

Cluster 0:
nova.osapi_compute.wsgi.server [req-b9718cd8-f6...
nova.osapi_compute.wsgi.server [req-4895c258-b2...
nova.osapi_compute.wsgi.server [req-ee8bc8ba-92...
nova.osapi_compute.wsgi.server [req-f0bffbc3-5a...
nova.osapi_compute.wsgi.server [req-2bf7cfee-a2...

Cluster 5:
nova.compute.claims [req-a07ac654-8e81-416d-bfb...
nova.compute.claims [req-d6986b54-3735-4a42-907...
nova.compute.claims [req-72b4858f-049e-49e1-b31...
nova.compute.claims [req-5c8f52bd-8e3c-41f0-95a...
nova.compute.claims [req-d38f479d-9bb9-4276-968...

Cluster 11:
User User685 logged out.
 User User395 logged in.
 User User225 logged in.
User User494 logged out.
 User User900 logged in.

Cluster 13:
Backup started at 2025-05-14 07:06:55.
Backup started at 2025-02-15 20:00:19.
  Backup ended at 2025-08-08 13:06:23.
Backup started at 2025-11-14 08:27:43.
Backup started at 2025-12-09 10:19:11.

Cluster 7:
Multiple bad login attempts detected on user 85...
Multiple login failures occurred on user 9052 a...
  User 

In [20]:
import re
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (in|out).": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern,label in regex_patterns.items():
        if re.search(pattern, log_message,re.IGNORECASE):
            return label
    return None
    

In [21]:
classify_with_regex("Backup 20:00:19.")

In [24]:
df['regex_label'] = df['log_message'].apply(classify_with_regex)

In [32]:
df_non_reg = df[df['regex_label'].isnull()].copy()
df_non_reg

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,NaN
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,NaN
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2,NaN
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,NaN
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0,NaN
...,...,...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert,0,NaN
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert,7,NaN
2407,2025-08-03 03:07:47,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert,0,NaN
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert,1,NaN


In [34]:
df_non_legacy = df_non_reg[df_non_reg.source!='LegacyCRM']
df_non_legacy.source.unique()

<StringArray>
['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem', 'ThirdPartyAPI']
Length: 5, dtype: str

In [35]:
filtered_embeddings = model.encode(df_non_legacy['log_message'].tolist())
filtered_embeddings[:2]

array([[-1.02939658e-01,  3.35459076e-02, -2.20260322e-02,
         1.55102310e-03, -9.86923371e-03, -1.78956211e-01,
        -6.34410307e-02, -6.01761937e-02,  2.81108450e-02,
         5.99620566e-02, -1.72618628e-02,  1.43368531e-03,
        -1.49560064e-01,  3.15292412e-03, -5.66030703e-02,
         2.71686260e-02, -1.49890119e-02, -3.54037806e-02,
        -3.62936333e-02, -1.45409480e-02, -5.61491353e-03,
         8.75538364e-02,  4.55120653e-02,  2.50963047e-02,
         1.00187324e-02,  1.24266408e-02, -1.39923513e-01,
         7.68695921e-02,  3.14095058e-02, -4.15245118e-03,
         4.36902493e-02,  1.71249919e-02, -8.00950900e-02,
         5.74005917e-02,  1.89092699e-02,  8.55261907e-02,
         3.96399312e-02, -1.34371817e-01, -1.44365174e-03,
         3.06709413e-03,  1.76854104e-01,  4.44887439e-03,
        -1.69274583e-02,  2.24267114e-02, -4.35049571e-02,
         6.09026384e-03, -9.98170208e-03, -6.23972826e-02,
         1.07372338e-02, -6.04890706e-03, -7.14661106e-0

In [36]:
x=filtered_embeddings
y=df_non_legacy['target_label']

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size =0.3,random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(x_train,y_train)
y_pred = clf.predict(x_test)
report = classification_report(y_test,y_pred)
print(report)


                precision    recall  f1-score   support

Critical Error       0.91      1.00      0.95        48
         Error       0.98      0.89      0.93        47
   HTTP Status       1.00      1.00      1.00       304
Resource Usage       1.00      1.00      1.00        49
Security Alert       1.00      0.99      1.00       123

      accuracy                           0.99       571
     macro avg       0.98      0.98      0.98       571
  weighted avg       0.99      0.99      0.99       571



In [37]:
import joblib
joblib.dump(clf, 'models/log_classifier.joblib')

['models/log_classifier.joblib']